# de-novo clustering of FM embeddings

each embedding space (Virchow2 H&E 1280d, Novae ST 64d, gpath2vec pathways 512d)  
is clustered independently with Leiden, then associated to two existing reference systems:

- **mc_megacluster** (NMF on ST counts, Wang 2024) — 94% coverage, but NMI=0.539 vs patient_id  
- **pathologist compartment** (Wang 2024 annotation) — 32% coverage, coarse categories  

neither reference is strong on its own. the FM clusters are the candidate better signal;  
the two references together give the best available biological interpretation.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import glob
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

sns.set_theme(style='ticks', palette='Set2', context='notebook')
sc.settings.verbosity = 0

In [ ]:
# --- config ---
NICHE_DIR   = Path('../../data/embeddings/niches_v3')
N_NEIGHBORS = 15
RESOLUTION  = 0.5
N_PCS_HE    = 50
N_PCS_GP    = 50
SEED        = 42

# mc_megacluster biological identities from Wang 2024 differential top genes
MC_NAMES = {
    1:  'CAF stroma',
    2:  'mc2',
    3:  'mc3',
    4:  'mc4',
    5:  'plasma cells',
    6:  'mc6',
    7:  'mc7',
    9:  'mc9',
    10: 'mc10',
    11: 'mc11',
    12: 'B-cell follicle / TLS',
    14: 'mc14',
}

## load

In [ ]:
files = sorted(NICHE_DIR.glob('*.parquet'))
print(f'{len(files)} subarrays')

dfs = []
for f in files:
    df = pd.read_parquet(f, columns=[
        'virchow2_niche', 'novae_niche', 'gpath2vec_niche',
        'mc_megacluster', 'compartment', 'patient_id', 'subarray'
    ])
    dfs.append(df)

meta = pd.concat(dfs).reset_index(drop=True)
mask = (
    meta['virchow2_niche'].notna() &
    meta['novae_niche'].notna() &
    meta['gpath2vec_niche'].notna()
)
meta = meta[mask].copy().reset_index(drop=True)

# label mc with names; -1 -> NaN
meta['mc_name'] = meta['mc_megacluster'].map(MC_NAMES)
meta['mc_label'] = meta['mc_megacluster'].apply(
    lambda x: f'mc{int(x)}: {MC_NAMES.get(int(x), "?")}' if pd.notna(x) and x != -1 else np.nan
)

print(f'{len(meta):,} niches   {meta["patient_id"].nunique()} patients   {meta["subarray"].nunique()} subarrays')
print(f'mc_megacluster labeled: {(meta["mc_megacluster"] != -1).sum():,}  ({(meta["mc_megacluster"] != -1).mean()*100:.0f}%)')
print(f'compartment labeled:    {meta["compartment"].notna().sum():,}  ({meta["compartment"].notna().mean()*100:.0f}%)')

X_he = np.vstack(meta['virchow2_niche'].values)
X_st = np.vstack(meta['novae_niche'].values)
X_gp = np.vstack(meta['gpath2vec_niche'].values)
print(f'\nH&E {X_he.shape}  ST {X_st.shape}  pathways {X_gp.shape}')

## cluster each embedding space

In [ ]:
def cluster_space(X, name, n_pcs=None, n_neighbors=15, resolution=0.5, seed=42):
    adata = sc.AnnData(X=X.copy())
    if n_pcs and X.shape[1] > n_pcs:
        sc.tl.pca(adata, n_comps=n_pcs, random_state=seed)
        use_rep = 'X_pca'
    else:
        adata.obsm['X_emb'] = X.copy()
        use_rep = 'X_emb'
    sc.pp.neighbors(adata, use_rep=use_rep, n_neighbors=n_neighbors, random_state=seed)
    sc.tl.leiden(adata, resolution=resolution, key_added='leiden', random_state=seed)
    sc.tl.umap(adata, random_state=seed)
    n = adata.obs['leiden'].nunique()
    print(f'{name:12s}: {n:3d} clusters')
    return adata

print(f'resolution={RESOLUTION}  k={N_NEIGHBORS}\n')
adata_he = cluster_space(X_he, 'H&E',      n_pcs=N_PCS_HE)
adata_st = cluster_space(X_st, 'ST',       n_pcs=None)
adata_gp = cluster_space(X_gp, 'pathways', n_pcs=N_PCS_GP)

for adata in [adata_he, adata_st, adata_gp]:
    adata.obs['mc_label']    = meta['mc_label'].values
    adata.obs['compartment'] = meta['compartment'].values
    adata.obs['patient_id']  = meta['patient_id'].astype(str).values

## UMAP — leiden / mc_megacluster / compartment

In [ ]:
modalities = [
    (adata_he, 'H&E (Virchow2 1280d)'),
    (adata_st, 'ST (Novae 64d)'),
    (adata_gp, 'pathways (gpath2vec 512d)'),
]

fig, axes = plt.subplots(3, 3, figsize=(15, 13))

for row, (adata, title) in enumerate(modalities):
    sc.pl.umap(adata, color='leiden', ax=axes[row, 0], show=False, frameon=False,
               title=f'{title}\nleiden', legend_loc='on data', legend_fontsize=6)
    sc.pl.umap(adata, color='mc_label', ax=axes[row, 1], show=False, frameon=False,
               title='mc_megacluster (94% coverage)', na_color='#e0e0e0',
               legend_loc='right margin', legend_fontsize=6)
    sc.pl.umap(adata, color='compartment', ax=axes[row, 2], show=False, frameon=False,
               title='compartment (32% coverage)', na_color='#e0e0e0',
               legend_loc='right margin', legend_fontsize=6)

plt.suptitle('de-novo FM clusters vs existing references', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

## association tables — cluster × mc and cluster × compartment

In [ ]:
def association_heatmaps(adata, modality_name):
    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, adata.obs['leiden'].nunique() * 0.4)))

    for ax, (ref_col, ref_label, threshold) in zip(axes, [
        ('mc_label',    'mc_megacluster (94%)',   None),
        ('compartment', 'compartment (32%)',       None),
    ]):
        labeled = adata.obs[adata.obs[ref_col].notna() & (adata.obs[ref_col] != '')]
        if len(labeled) == 0:
            ax.set_title(f'{ref_label}\nno labels')
            continue
        ct = pd.crosstab(labeled['leiden'], labeled[ref_col], normalize='index')
        # sort rows by dominant category
        ct = ct.loc[ct.idxmax(axis=1).sort_values().index]
        sns.heatmap(ct, ax=ax, cmap='YlOrRd', vmin=0, vmax=1,
                    linewidths=0.3, cbar_kws={'shrink': 0.5, 'label': 'fraction of cluster'},
                    xticklabels=True, yticklabels=True)
        ax.set_title(f'vs {ref_label}\n({len(labeled):,} labeled niches)')
        ax.set_xlabel('')
        ax.set_ylabel('leiden cluster')
        ax.tick_params(axis='x', rotation=45, labelsize=8)

    fig.suptitle(f'{modality_name} — cluster composition', fontsize=11)
    plt.tight_layout()
    plt.show()

for adata, name in modalities:
    association_heatmaps(adata, name)

## cluster annotation table — dominant mc + dominant compartment per cluster

In [ ]:
def annotation_table(adata, modality_name):
    obs = adata.obs.copy()
    rows = []
    for cluster in sorted(obs['leiden'].unique(), key=lambda x: int(x)):
        sub = obs[obs['leiden'] == cluster]
        n = len(sub)

        mc_sub = sub[sub['mc_label'].notna()]
        if len(mc_sub):
            top_mc = mc_sub['mc_label'].value_counts().idxmax()
            top_mc_frac = mc_sub['mc_label'].value_counts(normalize=True).max()
        else:
            top_mc, top_mc_frac = 'n/a', 0.0

        comp_sub = sub[sub['compartment'].notna() & (sub['compartment'] != '')]
        if len(comp_sub):
            top_comp = comp_sub['compartment'].value_counts().idxmax()
            top_comp_frac = comp_sub['compartment'].value_counts(normalize=True).max()
        else:
            top_comp, top_comp_frac = 'n/a', 0.0

        rows.append({
            'cluster': cluster, 'n': n,
            'dominant_mc': top_mc, 'mc_frac': round(top_mc_frac, 2),
            'dominant_compartment': top_comp, 'comp_frac': round(top_comp_frac, 2),
        })

    tbl = pd.DataFrame(rows).set_index('cluster')
    print(f'\n{modality_name}:')
    print(tbl.to_string())
    return tbl

tables = {}
for adata, name in modalities:
    tables[name] = annotation_table(adata, name)

## cross-modal ARI — do the three embedding spaces agree?

and how does each compare to mc_megacluster?

In [ ]:
he = adata_he.obs['leiden'].values
st = adata_st.obs['leiden'].values
gp = adata_gp.obs['leiden'].values

# mc labels where valid
mc_valid = meta['mc_megacluster'].values
mc_mask = (mc_valid != -1) & pd.notna(mc_valid)

print('ARI between FM embedding spaces:')
print(f'  H&E  vs ST:       {adjusted_rand_score(he, st):.4f}')
print(f'  H&E  vs pathways: {adjusted_rand_score(he, gp):.4f}')
print(f'  ST   vs pathways: {adjusted_rand_score(st, gp):.4f}')

print('\nARI vs mc_megacluster (where labeled):')
print(f'  H&E  vs mc: {adjusted_rand_score(he[mc_mask], mc_valid[mc_mask].astype(str)):.4f}')
print(f'  ST   vs mc: {adjusted_rand_score(st[mc_mask], mc_valid[mc_mask].astype(str)):.4f}')
print(f'  pathways vs mc: {adjusted_rand_score(gp[mc_mask], mc_valid[mc_mask].astype(str)):.4f}')

print('\nNMI vs patient_id:')
pid = meta['patient_id'].astype(str).values
for label, arr in [('H&E', he), ('ST', st), ('pathways', gp)]:
    nmi = normalized_mutual_info_score(arr, pid)
    flag = '  <- patient-confounded' if nmi > 0.5 else ''
    print(f'  {label:12s}: {nmi:.4f}{flag}')